In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load Dataset, Clean Missing Values & Feature Engineering
This cell handles the data preparation pipeline:
- Loads `housing.csv` into a pandas DataFrame.
- One-hot encodes the categorical column `ocean_proximity` using `pd.get_dummies(..., drop_first=True)`.
- Drops rows with null values in `total_bedrooms`.
- Engineers 3 new domain-specific ratio features:
  - `rooms_per_houshold`: Average rooms per household (`total_rooms / households`).
  - `bedrooms_per_rooms`: Proportion of bedrooms (`total_bedrooms / total_rooms`).
  - `people_per_houses`: Average household occupancy (`population / households`).

In [2]:
# Load the housing dataset from CSV file
df = pd.read_csv("housing.csv")

# Convert categorical column 'ocean_proximity' into one-hot encoded dummy variables
housing_data = pd.get_dummies(df, columns=["ocean_proximity"], drop_first=True)

# Drop rows containing missing/null values in the 'total_bedrooms' column
housing_data = housing_data.dropna(subset=["total_bedrooms"])

# Feature engineering: structural and density ratios
housing_data["rooms_per_houshold"] = housing_data["total_rooms"]/housing_data["households"]
housing_data["bedrooms_per_rooms"] = housing_data["total_bedrooms"]/housing_data["total_rooms"]
housing_data["people_per_houses"] = housing_data["population"]/housing_data["households"]

## 2. Separate Features and Split Data into Train/Test Sets
This cell:
- Separates independent features (`X`) by dropping `median_house_value`.
- Defines target vector (`y = median_house_value`).
- Splits data into 80% training and 20% testing sets using `random_state=404` for reproducible splits.

In [3]:
# Separate independent features (X) and target variable (y)
X = housing_data.drop(columns=["median_house_value"])
y = housing_data["median_house_value"]

# Split the data into training (80%) and testing (20%) sets
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2, random_state=404)

## 3. Define Evaluation Helper Function
Defines `evaluate_model(y_true, y_pred, model_name)` to print three evaluation metrics:
- **Mean Absolute Error (MAE)**: Average magnitude of prediction errors in dollars.
- **Root Mean Squared Error (RMSE)**: Square root of variance of residuals.
- **R-squared (R²)**: Proportion of variance in house prices explained by the model.

In [4]:
# Function to calculate and print key evaluation metrics (MAE, RMSE, R2)
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"Mean Absolute Error (MAE):      ${mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R-squared (R2) Score:           {r2:.4f}\n")

## 4. Train and Evaluate Linear Regression (Baseline Model)
This cell:
- Imports `LinearRegression` from `sklearn.linear_model`.
- Fits the model on `X_train` and `y_train`.
- Generates predictions on `X_test`.
- Prints evaluation metrics as the baseline benchmark.

In [5]:
from sklearn.linear_model import LinearRegression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

evaluate_model(y_test, y_pred, "Linear Regression")

--- Linear Regression ---
Mean Absolute Error (MAE):      $48,680.38
Root Mean Squared Error (RMSE): $66,819.12
R-squared (R2) Score:           0.6600



## 5. Train and Evaluate Random Forest Regressor
This cell:
- Imports `RandomForestRegressor` from `sklearn.ensemble`.
- Initializes the model with `random_state=42` and `n_jobs=-1` (parallel processing on all CPU cores).
- Trains on `(X_train, y_train)` and predicts on `X_test`.
- Evaluates performance to compare against the linear baseline.

In [6]:
from sklearn.ensemble import RandomForestRegressor
forest_model = RandomForestRegressor(
    random_state = 42,
    n_jobs = -1
)
forest_model.fit(X_train, y_train)
forest_prediction = forest_model.predict(X_test)

evaluate_model(y_test, forest_prediction, "Random Forest Regressor")

--- Random Forest Regressor ---
Mean Absolute Error (MAE):      $32,031.82
Root Mean Squared Error (RMSE): $48,732.27
R-squared (R2) Score:           0.8191

